# TP 2 — Profiling & périmètre · NutriScope

**Date :** 25/08/2026  
**Équipe :** Sacha & Clément

## Objectif
Profiler les données du TP1, identifier les colonnes utiles, mesurer les anomalies, comparer les catégories et décider du périmètre NutriScope.

### Répartition
- **Clément :** données, DuckDB, cardinalités, doublons, nutriments, unités, valeurs impossibles.
- **Sacha :** inventaire des colonnes, catégories, images, substitution, assistant.
- **Ensemble :** périmètre final, seuil de complétude et décisions.

> Le TP2 observe et documente. Le nettoyage industrialisé sera réalisé au TP9.


## 0. Imports et configuration

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/food.parquet")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/food.parquet")

assert DATA_PATH.exists(), f"Fichier introuvable : {DATA_PATH}"
con = duckdb.connect()
print(DATA_PATH)


# 1. Population de référence — Clément

Le Parquet contient environ 4,6 M de produits et 111 colonnes. DuckDB est utilisé pour les requêtes volumineuses.


In [ ]:
dimensions = con.execute(f'''
SELECT COUNT(*) AS nb_lignes
FROM read_parquet('{DATA_PATH}')
''').df()
dimensions


In [ ]:
schema = con.execute(f'''
DESCRIBE SELECT * FROM read_parquet('{DATA_PATH}')
''').df()
print(f"Nombre de colonnes : {len(schema)}")
display(schema)


In [ ]:
france_stats = con.execute(f'''
SELECT
    COUNT(*) AS nb_produits_france,
    COUNT(*) FILTER (WHERE obsolete = TRUE) AS nb_obsoletes
FROM read_parquet('{DATA_PATH}')
WHERE 'en:france' IN countries_tags
''').df()
france_stats


# 2. Profiling général — Clément

## 2.1 Cardinalités et valeurs manquantes

In [ ]:
important_columns = [
    "code", "product_name", "brands", "brands_tags",
    "categories_tags", "countries_tags", "quantity",
    "serving_size", "nutriscore_grade", "nutriscore_score",
    "nova_group", "environmental_score_grade",
    "images", "nutriments", "completeness"
]

queries = []
for col in important_columns:
    queries.append(f'''
    SELECT
        '{col}' AS colonne,
        COUNT(*) AS lignes,
        COUNT(*) FILTER (WHERE "{col}" IS NULL) AS nulls,
        COUNT(DISTINCT "{col}") AS cardinalite
    FROM read_parquet('{DATA_PATH}')
    WHERE 'en:france' IN countries_tags
    ''')

profiling = con.execute(" UNION ALL ".join(queries)).df()
profiling["taux_null_pct"] = profiling["nulls"] / profiling["lignes"] * 100
profiling.sort_values("taux_null_pct")


## 2.2 Doublons de codes-barres

In [ ]:
duplicate_stats = con.execute(f'''
WITH codes AS (
    SELECT code, COUNT(*) AS n
    FROM read_parquet('{DATA_PATH}')
    WHERE 'en:france' IN countries_tags
      AND code IS NOT NULL
      AND TRIM(code) <> ''
    GROUP BY code
)
SELECT
    COUNT(*) AS nb_codes_uniques,
    COUNT(*) FILTER (WHERE n > 1) AS nb_codes_dupliques,
    COALESCE(SUM(n) FILTER (WHERE n > 1), 0) AS nb_lignes_concernees
FROM codes
''').df()
duplicate_stats


In [ ]:
duplicate_examples = con.execute(f'''
SELECT code, COUNT(*) AS n
FROM read_parquet('{DATA_PATH}')
WHERE 'en:france' IN countries_tags
  AND code IS NOT NULL
  AND TRIM(code) <> ''
GROUP BY code
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 20
''').df()
duplicate_examples


# 3. Profiling des catégories — Sacha

`categories_tags` est une liste de tags. Une fiche pouvant appartenir à plusieurs catégories, les volumes par catégorie ne sont pas nécessairement additifs.


In [ ]:
top_categories = con.execute(f'''
SELECT category, COUNT(*) AS nb_produits
FROM read_parquet('{DATA_PATH}'),
     UNNEST(categories_tags) AS t(category)
WHERE 'en:france' IN countries_tags
  AND category IS NOT NULL
GROUP BY category
ORDER BY nb_produits DESC
LIMIT 50
''').df()
top_categories.head(20)


In [ ]:
top_categories.head(20).plot(
    kind="barh", x="category", y="nb_produits",
    figsize=(10, 8), legend=False
)
plt.xlabel("Nombre de produits")
plt.ylabel("Catégorie")
plt.title("Top 20 des catégories — France")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
category_quality = con.execute(f'''
SELECT
    category,
    COUNT(*) AS nb_produits,
    AVG(CASE WHEN nutriscore_grade IS NOT NULL
              AND LOWER(TRIM(nutriscore_grade)) NOT IN ('', 'unknown', 'not-applicable')
             THEN 1.0 ELSE 0.0 END) * 100 AS pct_nutriscore,
    AVG(CASE WHEN nutriments IS NOT NULL AND len(nutriments) > 0
             THEN 1.0 ELSE 0.0 END) * 100 AS pct_nutrition,
    AVG(CASE WHEN images IS NOT NULL AND len(images) > 0
             THEN 1.0 ELSE 0.0 END) * 100 AS pct_images,
    AVG(CASE WHEN product_name IS NOT NULL AND len(product_name) > 0
             THEN 1.0 ELSE 0.0 END) * 100 AS pct_nom,
    AVG(CASE WHEN brands IS NOT NULL AND TRIM(brands) <> ''
             THEN 1.0 ELSE 0.0 END) * 100 AS pct_marque
FROM read_parquet('{DATA_PATH}'),
     UNNEST(categories_tags) AS t(category)
WHERE 'en:france' IN countries_tags
  AND category IS NOT NULL
GROUP BY category
HAVING COUNT(*) >= 1000
ORDER BY nb_produits DESC
''').df()
category_quality.head(30)


# 4. Profiling nutritionnel — Clément

`nutriments` est une liste de structures. On extrait les valeurs pour 100 g.


In [ ]:
nutrient_names = [
    "energy", "sugars", "salt", "proteins",
    "carbohydrates", "fat", "saturated-fat", "fiber"
]
nutrient_sql = ", ".join(f"'{x}'" for x in nutrient_names)

nutrients = con.execute(f'''
SELECT
    code,
    n.name AS nutrient,
    TRY_CAST(n."100g" AS DOUBLE) AS value_100g,
    n.unit AS unit
FROM read_parquet('{DATA_PATH}'),
     UNNEST(nutriments) AS t(n)
WHERE 'en:france' IN countries_tags
  AND n.name IN ({nutrient_sql})
''').df()

nutrients.head()


In [ ]:
nutrient_completeness = (
    nutrients.assign(has_value=nutrients["value_100g"].notna())
    .groupby("nutrient")
    .agg(produits=("code", "nunique"), renseignes=("has_value", "sum"))
)
nutrient_completeness["taux_renseigne_pct"] = (
    nutrient_completeness["renseignes"] /
    nutrient_completeness["produits"] * 100
)
nutrient_completeness.sort_values("taux_renseigne_pct", ascending=False)


## 4.1 Nutriments clés : energy, sugars, salt

In [ ]:
key = nutrients[nutrients["nutrient"].isin(["energy", "sugars", "salt"])]
key_pivot = key.pivot_table(
    index="code", columns="nutrient",
    values="value_100g", aggfunc="first"
)
for col in ["energy", "sugars", "salt"]:
    if col not in key_pivot:
        key_pivot[col] = np.nan

summary_key = pd.DataFrame({
    "mesure": [
        "Produits avec energy",
        "Produits avec sugars",
        "Produits avec salt",
        "Produits avec les 3"
    ],
    "nombre": [
        key_pivot["energy"].notna().sum(),
        key_pivot["sugars"].notna().sum(),
        key_pivot["salt"].notna().sum(),
        key_pivot[["energy", "sugars", "salt"]].notna().all(axis=1).sum()
    ]
})
summary_key


## 4.2 Valeurs impossibles

On quantifie les anomalies sans les supprimer.

In [ ]:
rules = {
    "sugars": (0, 100),
    "salt": (0, 100),
    "proteins": (0, 100),
    "carbohydrates": (0, 100),
    "fat": (0, 100),
    "saturated-fat": (0, 100),
    "fiber": (0, 100),
}

rows = []
for nutrient, (lo, hi) in rules.items():
    s = nutrients.loc[nutrients["nutrient"] == nutrient, "value_100g"]
    rows.append({
        "nutriment": nutrient,
        "valeurs_sous_minimum": int((s < lo).sum()),
        "valeurs_au_dessus_maximum": int((s > hi).sum())
    })

energy = nutrients.loc[nutrients["nutrient"] == "energy", "value_100g"]
rows.append({
    "nutriment": "energy",
    "valeurs_sous_minimum": int((energy < 0).sum()),
    "valeurs_au_dessus_maximum": np.nan,
    "valeurs_nulles": int((energy == 0).sum())
})

pd.DataFrame(rows)


In [ ]:
anomalies = nutrients[
    (
        nutrients["nutrient"].isin(rules)
        & (
            (nutrients["value_100g"] < 0) |
            (nutrients["value_100g"] > 100)
        )
    )
    |
    (
        (nutrients["nutrient"] == "energy") &
        (
            (nutrients["value_100g"] < 0) |
            (nutrients["value_100g"] == 0)
        )
    )
]
anomalies.head(30)


## 4.3 Unités

In [ ]:
units = (
    nutrients.groupby(["nutrient", "unit"], dropna=False)
    .size()
    .reset_index(name="nb_occurrences")
    .sort_values(["nutrient", "nb_occurrences"], ascending=[True, False])
)
units


# 5. Nutri-Score — Clément

In [ ]:
nutriscore_stats = con.execute(f'''
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (
        WHERE nutriscore_grade IS NOT NULL
          AND LOWER(TRIM(nutriscore_grade))
              NOT IN ('', 'unknown', 'not-applicable')
    ) AS renseigne
FROM read_parquet('{DATA_PATH}')
WHERE 'en:france' IN countries_tags
''').df()

nutriscore_stats["part_renseignee_pct"] = (
    nutriscore_stats["renseigne"] /
    nutriscore_stats["total"] * 100
)
nutriscore_stats


In [ ]:
nutriscore_distribution = con.execute(f'''
SELECT
    LOWER(TRIM(nutriscore_grade)) AS nutriscore_grade,
    COUNT(*) AS nb_produits
FROM read_parquet('{DATA_PATH}')
WHERE 'en:france' IN countries_tags
GROUP BY 1
ORDER BY nb_produits DESC
''').df()
nutriscore_distribution


# 6. Images — Sacha

In [ ]:
image_stats = con.execute(f'''
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (
        WHERE images IS NOT NULL AND len(images) > 0
    ) AS avec_image
FROM read_parquet('{DATA_PATH}')
WHERE 'en:france' IN countries_tags
''').df()
image_stats["pct_avec_image"] = image_stats["avec_image"] / image_stats["total"] * 100
image_stats


# 7. Inventaire des colonnes — Sacha

S'appuyer sur `data-fields.txt` et `food_parquet_structure.md`.

Chaque colonne doit être classée :
- ESSENTIELLE
- UTILE
- OPTIONNELLE
- BRUIT / TECHNIQUE
- À ÉTUDIER

Fonctionnalités : produit, nutrition, Nutri-Score, substitution, images, assistant.


In [ ]:
column_inventory = pd.DataFrame([
    # ("code", "Produit", "ESSENTIELLE", "Identifiant produit"),
    # ("product_name", "Produit", "ESSENTIELLE", "Nom affiché"),
    # ("brands", "Produit", "UTILE", "Marque"),
    # ("categories_tags", "Produit/Substitution", "ESSENTIELLE", "Catégorisation"),
    # ("nutriments", "Nutrition/ML", "ESSENTIELLE", "Variables nutritionnelles"),
    # ("nutriscore_grade", "Nutri-Score", "ESSENTIELLE", "Score A-E"),
    # ("images", "Images", "ESSENTIELLE", "Futur classifieur"),
    # ("creator", "Technique", "BRUIT / TECHNIQUE", "Métadonnée contributeur"),
], columns=["colonne", "fonction", "decision", "justification"])
column_inventory


# 8. Images, substitution et assistant — Sacha

À compléter à partir des résultats précédents.

### Images
Évaluer la présence d'images par catégorie.

### Substitution
Vérifier la disponibilité conjointe de :
`categories_tags`, `nutriments`, `nutriscore_grade`, `brands_tags`, `labels_tags`.

### Assistant
Vérifier la disponibilité de :
`code`, `product_name`, `generic_name`, `brands`, `categories_tags`,
`nutriments`, `nutriscore_grade`, `ingredients_text`, `allergens_tags`, `labels_tags`.


# 9. Seuil de complétude — Sacha + Clément

Tester plusieurs seuils (50 %, 60 %, 70 %, 80 %, 90 %) selon les champs retenus.

**À décider ensemble :**
- définition d'un produit exploitable ;
- seuil minimal ;
- impact sur le volume global et par catégorie.


In [ ]:
# TODO : construire ici le score de complétude final après l'inventaire des colonnes.
# Exemple :
#
# completion_fields = [...]
# completion_score = nombre_de_champs_renseignes / nombre_de_champs_attendus
#
# Puis comparer les seuils 0.50, 0.60, 0.70, 0.80 et 0.90.


# 10. Décision de périmètre — Sacha + Clément

Le TP demande **5 à 8 catégories**.

| Catégorie | Volume | Nutrition | Nutri-Score | Images | Substitution | Décision |
|---|---:|---:|---:|---:|---:|---|
| | | | | | | |
| | | | | | | |

## Catégories retenues

1. 
2. 
3. 
4. 
5. 
6. 
7. 
8. 

## Seuil de complétude retenu

**XX %**

## Justification

À rédiger à partir des mesures obtenues.


# 11. Conclusions et préparation du TP9

## Constats principaux
- 

## Anomalies à traiter au TP9
- 

## Colonnes conservées
- 

## Colonnes écartées
- 

## Périmètre final
- 

## Seuil de complétude
- 

## Points à challenger avec l'autre équipe
- 


# 12. Checklist finale

- [ ] Profiling général terminé
- [ ] Doublons analysés
- [ ] Nutriments clés analysés
- [ ] Valeurs impossibles quantifiées
- [ ] Unités analysées
- [ ] Inventaire des colonnes terminé
- [ ] Catégories comparées
- [ ] Images analysées
- [ ] Seuil de complétude mesuré
- [ ] 5 à 8 catégories choisies
- [ ] Décisions validées par Sacha et Clément
- [ ] `docs/perimetre.md` rédigé
- [ ] Relecture croisée effectuée
